# H&M M5 — arm B: M3 두 축 그래프 + N/V 분리 M4 손실 (개발 seed 43)

두 arm(A 원형 M4, B N/V 분리)은 **같은 M3 두 축(V+N) 중심화 가치그래프**를 쓰고 BPR 학습행 가중치만 다릅니다. 이 노트북은 **arm B 하나만** 학습합니다 — 다른 Colab 런타임에서 arm A 노트북을 동시에 돌리면 됩니다.

- A: `1 + 0.5·q_C·상품금액백분위·고객가격대적합도` (확증된 H&M M4와 동일, 가중치 감사값 일치 확인 후 학습)
- B: `(1 + 0.5·q_N) × (1 + 0.5·q_V·상품금액백분위·적합도)`

고정: H&M 개발분할(2020-09-02~08), seed 43, **100 epoch까지 학습 후 판정**(학습 상태는 Drive에 저장 — 같은 코드로 나중에 300까지 이어갈 수 있음). 얼리스탑핑·epoch 선택 없음. test/holdout은 열지 않습니다.

판정(실행 전 고정, epoch 100): ① M1 대비 가격·구매금액 가중 적중값@10·가중 NDCG@10 모두 상승 ② 여섯 정확도 ≥ M1의 99% ③ 경제2지표@10 ≥ M4의 99%. M1·M3·M4는 기존 seed 43 결과 재사용(재학습 없음). 단일 개발시드라 유의성·일반화는 주장하지 않습니다.

**런타임:** GPU + **고용량 RAM** 권장(H&M 전체 2,800만 행). 예상 약 12~13시간(H&M M4 기록 epoch당 약 450초). 끊기면 같은 셀을 다시 실행하면 마지막 epoch부터 이어집니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
REFS = [ROOT/'results_v3_hm_clv_m3_centered_value_graph_s43_v1', ROOT/'results_v3_hm_m4_k1_assignment_control_hm2y_development_screen_v1']
if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습을 시작하지 않았습니다.') from exc
for ref in REFS:
    if not list(ref.glob('*.json')):
        raise RuntimeError(f'기준 결과가 없습니다: {ref} — 재학습하지 않고 중단합니다.')

SOURCE_COMMIT = '7dcdf99e5924b93545c231dec82fb900f7e453b1'
VARIANT = 'split_nv'   # arm B
REPO = Path('/content/clv-m5-hm2y-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
for name in ('lightgcn_clv_v3', 'lightgcn_clv_m3_centered_value_graph_hm2y', 'clv_m5_m3_m4_split_nv_hm2y_screen'):
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent != REPO.resolve():
        raise RuntimeError(f'{name}이 다른 코드에서 로드됐습니다. 런타임을 다시 시작하세요.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_m3_m4_split_nv_screen
clv_m5_m3_m4_split_nv_screen.self_test()
import clv_m5_m3_m4_split_nv_hm2y_screen as screen
cfg = screen.configure()
print(json.dumps({'source_commit': SOURCE_COMMIT, 'arm': VARIANT, 'model_id': screen.ARMS[VARIANT],
                  'judge_epoch': screen.JUDGE_EPOCH, 'out_dir': cfg.out_dir}, ensure_ascii=False, indent=2))

## 데이터 준비 → 학습 전 점검 → 학습

준비 단계에서 ① 그래프 강도 β가 H&M M3 기준(0.6088)과 같은지 ② M1·M3·M4 기준 결과가 같은 설정인지, 두 실행의 M1@100이 같은지 ③ A의 가중치가 확증된 H&M M4 감사값(평균 원시가중 1.21998, CV 0.15905)과 같은지 확인하고, 하나라도 다르면 학습하지 않고 멈춥니다.

In [ ]:
prepared = screen.prepare(cfg)
arm = screen.train_arm(cfg, prepared, VARIANT, stop_epoch=screen.JUDGE_EPOCH)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
result = screen.report(cfg, prepared)   # 끝난 arm만 비교 (다른 arm이 아직이면 이 arm만)
key = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
       'price_purchase_amount_weighted_hit@10', 'vndcg@10',
       'price_purchase_amount_weighted_hit@20', 'vndcg@20',
       'price_purchase_amount_weighted_hit@50', 'vndcg@50']
view = result['comparison']
view = view[view.metric.isin(key)]
for (epoch, model, ref), part in view.groupby(['epoch', 'model_id', 'reference']):
    print(f'\n=== epoch {epoch} | {model} vs {ref} ===')
    print(part[['metric', 'reference_value', 'candidate_value', 'ratio']].to_string(index=False))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
zip_path = Path('/content/clv_m5_m3_m4_split_nv_hm2y_s43_split_nv_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))